# SurvivalLLM: Offline Survival Q&A Fine-Tuning

Generate a survival Q&A dataset with the LightningRod SDK and fine-tune with [Tinker](https://tinker-docs.thinkingmachines.ai/) LoRA for offline deployment.

**Pipeline:** [Pluto](https://github.com/redotvideo/pluto) topic tree → LightningRod Q&A generation → Tinker SFT

In [ ]:
%pip install lightningrod-ai python-dotenv pluto-data pandas tinker -q

from IPython.display import clear_output
clear_output()

## Setup

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) for your API key and **$50 of free credits**.

Also set `OPENROUTER_API_KEY` (for Pluto) and `TINKER_API_KEY` (for training).

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
lr = LightningRod(api_key=config.get_config_value("LIGHTNINGROD_API_KEY"))

## 1. Generate survival topics

Use Pluto to build a hierarchical topic tree. Each root-to-leaf path becomes a seed for Q&A generation.

In [3]:
DOMAINS = [
    "Field medicine and trauma care in austere environments",
    "Water purification and safe water sourcing without electricity",
    "Food preservation, canning, and long-term storage without refrigeration",
    "Ham radio and emergency communications setup and operation",
    "Land navigation using map, compass, and natural indicators",
    "Growing food: gardening, permaculture, and seed saving",
    "Herbal medicine and natural remedies from wild plants",
    "Construction, structural repair, and improvised building",
    "Welding, metalworking, and tool fabrication",
    "Vehicle repair and mechanical troubleshooting without a shop",
    "Fire starting, fire management, and fuel sourcing",
    "Emergency shelter building from natural and salvaged materials",
    "Hunting, trapping, fishing, and wild game processing",
    "Knot tying, rope work, and cordage making",
    "Weather reading and natural forecasting without instruments",
    "Perimeter security, self-defense, and community safety planning",
]

In [ ]:
from pluto import TopicTree, TopicTreeArguments
from tqdm import tqdm

# degree=5, depth=3 → 125 paths/domain → 2000 total topics → 20k questions at 10/seed
# Start small for testing; scale up DOMAINS[:] and tree params for production runs.
TREE_DEGREE = 3
TREE_DEPTH = 2

topics = []
for domain in tqdm(DOMAINS[:2], desc="Building topic trees"):
    args = TopicTreeArguments(
        root_prompt=domain,
        tree_degree=TREE_DEGREE,
        tree_depth=TREE_DEPTH,
        model_system_prompt=(
            "You are an expert in survival and self-reliance. "
            "Generate specific, practical subtopics useful in a grid-down emergency."
        ),
    )
    tree = TopicTree(args)
    tree.build_tree(model_name="openrouter/google/gemini-3-flash-preview")
    topics.extend(" → ".join(path) for path in tree.tree_paths)

print(f"{len(topics)} topics")
for t in topics[:3]:
    print(f"  {t}")

## 2. Generate Q&A dataset

Upload topics as seeds, generate questions with `QuestionGenerator`, and label answers with `WebSearchLabeler` for web-grounded accuracy.

In [ ]:
from lightningrod import create_sample

samples = [create_sample(seed_text=t) for t in topics]
input_ds = lr.datasets.create_from_samples(samples)
print(f"Uploaded {input_ds.num_rows} seeds")

In [ ]:
from lightningrod import (
    FreeResponseAnswerType, QuestionGenerator,
    QuestionPipeline, WebSearchLabeler,
)

answer_type = FreeResponseAnswerType(
    labeler_instruction=(
        "You are a survival expert giving emergency field instructions. "
        "Give direct, numbered step-by-step instructions. No introductions, disclaimers, "
        "or filler. Start with the first action. Use specific measurements and techniques. "
        "Assume no professional help, stores, or infrastructure available."
    ),
    answer_format_instruction=(
        "Provide a direct, step-by-step survival answer. No introduction — start with step 1. "
        "Use specific measurements and techniques. Provide your answer between <answer></answer> tags."
    ),
    question_generation_instruction=(
        "Generate specific, practical how-to questions about survival techniques for "
        "grid-down emergencies. Each question must ask HOW to perform a specific procedure "
        "with limited or no modern tools. Each must cover a UNIQUE technique."
    ),
)

pipeline = QuestionPipeline(
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        questions_per_seed=10,
        instructions=(
            "Generate practical, actionable survival questions for grid-down emergencies. "
            "Questions must be specific, scenario-based, and ask HOW to do something with "
            "limited or no modern tools. Each must cover a DISTINCT technique."
        ),
        examples=[
            "How do I purify water using only sand, gravel, and charcoal when I have no commercial filter?",
            "What are the signs of a tension pneumothorax and how do I perform a needle decompression in the field?",
            "How do I build a Dakota fire hole to minimize visible smoke and maximize heat efficiency?",
            "How do I make a basic antenna for a Baofeng UV-5R to extend its range in mountainous terrain?",
        ],
        bad_examples=[
            "What is survival? (too vague)",
            "Tell me about water purification. (not actionable)",
            "How does a ham radio work? (theoretical, not a how-to)",
        ],
    ),
    labeler=WebSearchLabeler(answer_type=answer_type, confidence_threshold=0.8),
)

dataset = lr.transforms.run(pipeline, input_dataset=input_ds, name="SurvivalLLM")

result_samples = dataset.download()
valid = sum(1 for s in result_samples if s.is_valid)
print(f"{len(result_samples)} samples, {valid} valid ({valid/len(result_samples)*100:.0f}%)")

In [7]:
import pandas as pd

rows = [
    {
        "question": s.question.question_text if s.question else None,
        "answer": s.label.label if s.label else None,
        "confidence": s.label.label_confidence if s.label else None,
    }
    for s in result_samples if s.is_valid
]
pd.DataFrame(rows).head()

,question,answer,confidence
0,How do I find and safely tap a birch or maple ...,Identify a healthy tree at least 10 inches in ...,1.0
1,How do I create a primitive slow-sand filtrati...,"To create a slow-sand filter, place a layer of...",1.0
2,How do I create a rigid splint for a forearm f...,Wrap the forearm in a thick layer of padding l...,1.0
3,How do I incorporate crushed charcoal into a n...,Pulverize hardwood charcoal from a cold campfi...,1.0
4,How do I improvise a rigid splint for a fractu...,Tightly roll or fold the magazine around the f...,1.0


## 3. Build SFT training data

Convert to conversation format and save as JSONL.

In [ ]:
import json
from pathlib import Path

SYSTEM_PROMPT = (
    "You are SurvivalLLM. Give direct, step-by-step survival instructions. "
    "No introductions or disclaimers. Start with the first action. "
    "Be specific with measurements and techniques."
)

sft_data = []
for s in result_samples:
    if not s.is_valid:
        continue
    q = s.question.question_text if s.question else None
    a = s.label.label if s.label else None
    if not q or not a or a == "undetermined":
        continue
    sft_data.append({"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]})

sft_path = Path("survival_sft.jsonl")
sft_path.write_text("\n".join(json.dumps(r) for r in sft_data))
print(f"{len(sft_data)} training examples \u2192 {sft_path}")

## 4. Fine-tune with Tinker

LoRA SFT on the generated dataset. Saves a base model sampler before training for comparison.

In [ ]:
import random
import numpy as np
import tinker
from tinker import types

BASE_MODEL = "meta-llama/Llama-3.1-8B"
LEARNING_RATE = 2e-4

service = tinker.ServiceClient()
trainer = service.create_lora_training_client(base_model=BASE_MODEL, train_unembed=False)
tokenizer = trainer.get_tokenizer()


def tokenize_chat(messages, **kwargs) -> list[int]:
    result = tokenizer.apply_chat_template(messages, tokenize=True, **kwargs)
    return result.input_ids if hasattr(result, "input_ids") else list(result)


def conversation_to_datum(messages: list[dict]) -> types.Datum:
    full_tokens = tokenize_chat(messages)
    prefix_len = len(tokenize_chat(messages[:-1], add_generation_prompt=True))
    weights = [0.0] * prefix_len + [1.0] * (len(full_tokens) - prefix_len)
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=full_tokens[:-1]),
        loss_fn_inputs=dict(target_tokens=full_tokens[1:], weights=weights[1:]),
    )


datums = [conversation_to_datum(c["messages"]) for c in sft_data]
print(f"{len(datums)} datums tokenized")

In [ ]:
base_sampler = trainer.save_weights_and_get_sampling_client()

adam = tinker.AdamParams(learning_rate=LEARNING_RATE)
random.shuffle(datums)

NUM_EPOCHS = 3
for epoch in range(NUM_EPOCHS):
    result = trainer.forward_backward(datums, loss_fn="cross_entropy").result()
    trainer.optim_step(adam).result()
    logprobs = np.concatenate([o["logprobs"].tolist() for o in result.loss_fn_outputs])
    w = np.concatenate([d.loss_fn_inputs["weights"].tolist() for d in datums])
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS} — loss: {-np.dot(logprobs, w) / w.sum():.4f}")

ft_sampler = trainer.save_weights_and_get_sampling_client()

## 5. Evaluate: base vs fine-tuned

In [ ]:
def sample_answer(sampler, question):
    tokens = tokenize_chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ], add_generation_prompt=True)
    resp = sampler.sample(
        types.ModelInput.from_ints(tokens),
        num_samples=1,
        sampling_params=tinker.SamplingParams(max_tokens=512, temperature=0.7),
    )
    return tokenizer.decode(resp.result().sequences[0].tokens, skip_special_tokens=True)


test_questions = [
    "How do I purify water using only materials I can find in a forest?",
    "How do I stop severe arterial bleeding with no medical supplies?",
    "How do I start a fire in wet conditions with no matches?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    print(f"BASE: {sample_answer(base_sampler, q)[:300]}")
    print(f"FINE-TUNED: {sample_answer(ft_sampler, q)[:300]}")